### #8

Kaggle competition: [\[link\]]([link](https://www.kaggle.com/competitions/playground-series-s5e8/))

Entry by Robin R.P.M. Kras

robkras.com

### ⭐ 1. Introduction & Overview


Your Goal: Your goal is to predict whether a client will subscribe to a bank term deposit.

### 🔹 2. Import Libraries & Set Up


In [87]:
# =============================================================================    
# MACHINE LEARNING LIBRARIES - SIMPLE IMPORTS
# =============================================================================    

# Set environment variable for scipy array API support
import os
os.environ['SCIPY_ARRAY_API'] = '1'

# Core Data Science Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# Gradient Boosting Libraries
import xgboost as xgb
import lightgbm as lgb

# Deep Learning
#import torch
#import torch.nn as nn
#import torch.optim as optim
#import torchvision.transforms as transforms

import tensorflow as tf
from tensorflow import keras

# Visualization
import plotly.express as px
import plotly.graph_objects as go
from bokeh.plotting import figure, show, output_notebook
from bokeh.models import HoverTool

# Computer Vision
import cv2

# Scientific Computing & Statistics
import scipy.stats as stats
from scipy import optimize
import statsmodels.api as sm
from statsmodels.tsa.seasonal import seasonal_decompose

# Image Processing
from PIL import Image, ImageDraw, ImageFont

# Sampling and Resampling - Try import, use alternatives if failed
try:
    from imblearn.over_sampling import SMOTE
    from imblearn.under_sampling import RandomUnderSampler
    IMBLEARN_AVAILABLE = True
except ImportError:
    print("imblearn not available, using sklearn class_weight='balanced' instead")
    IMBLEARN_AVAILABLE = False

# Utilities
import sys
import warnings
import datetime
from pathlib import Path
import pickle
import json

# Configuration
plt.rcParams['figure.figsize'] = (10, 6)
sns.set_palette("husl")
warnings.filterwarnings('ignore')
SEED = 42

In [88]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

In [89]:
train.head()

,id,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,0,42,technician,married,secondary,no,7,no,no,cellular,25,aug,117,3,-1,0,unknown,0
1,1,38,blue-collar,married,secondary,no,514,no,no,unknown,18,jun,185,1,-1,0,unknown,0
2,2,36,blue-collar,married,secondary,no,602,yes,no,unknown,14,may,111,2,-1,0,unknown,0
3,3,27,student,single,secondary,no,34,yes,no,unknown,28,may,10,2,-1,0,unknown,0
4,4,26,technician,married,secondary,no,889,yes,no,cellular,3,feb,902,1,-1,0,unknown,1


In [90]:
train['balance_duration'] = train['balance'] * train['duration']  
train['campaign_previous'] = train['campaign'] * train['previous']
train['age_balance'] = train['age'] * train['balance']

test['balance_duration'] = test['balance'] * test['duration']
test['campaign_previous'] = test['campaign'] * test['previous']
test['age_balance'] = test['age'] * test['balance']

In [91]:
# One-Hot Encoding for categorical variables
from sklearn.preprocessing import OneHotEncoder
import pandas as pd

# Define categorical columns to encode
to_be_onehot_encoded = ['job', 'marital', 'default', 'housing', 'loan', 'contact', 'month', 
'poutcome']

# Method 1: Using pandas get_dummies (simpler)
train_encoded = pd.get_dummies(train, columns=to_be_onehot_encoded, drop_first=True)        
test_encoded = pd.get_dummies(test, columns=to_be_onehot_encoded, drop_first=True)

# Ensure test has same columns as train
test_encoded = test_encoded.reindex(columns=train_encoded.columns, fill_value=0)

# Update train and test
train = train_encoded
test = test_encoded

print(f"Original features: {len(to_be_onehot_encoded)}")
print(f"After one-hot encoding: {train.shape[1]} total features")

# Check for remaining object columns
remaining_objects = train.select_dtypes(include=['object']).columns.tolist()
print(f"Remaining object columns: {remaining_objects}")

# If any remain, handle them
if remaining_objects:
    from sklearn.preprocessing import LabelEncoder
    le = LabelEncoder()
    for col in remaining_objects:
        train[col] = le.fit_transform(train[col].astype(str))
        test[col] = le.transform(test[col].astype(str))
    print(f"Label encoded remaining columns: {remaining_objects}")
else:
    print("No remaining object columns to encode")

Original features: 8
After one-hot encoding: 45 total features
Remaining object columns: ['education']
Label encoded remaining columns: ['education']


In [92]:
train.head(3)

,id,age,education,balance,day,duration,campaign,pdays,previous,y,...,month_jul,month_jun,month_mar,month_may,month_nov,month_oct,month_sep,poutcome_other,poutcome_success,poutcome_unknown
0,0,42,1,7,25,117,3,-1,0,0,...,False,False,False,False,False,False,False,False,False,True
1,1,38,1,514,18,185,1,-1,0,0,...,False,True,False,False,False,False,False,False,False,True
2,2,36,1,602,14,111,2,-1,0,0,...,False,False,False,True,False,False,False,False,False,True


In [93]:
X = train.drop(columns=['id', 'y'])
y = train['y']

In [94]:
import optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

X_test = test.drop(columns=['id'])

def objective(trial):
    params = {
        'objective': 'binary:logistic',
        'eval_metric': 'auc',
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'random_state': SEED,
        'tree_method': 'gpu_hist',  # Use GPU acceleration if available
        'gpu_id': 0,  # Specify GPU ID if multiple GPUs are available
        'early_stopping_rounds': 50
    }

    # 3-fold CV for faster optimization
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
    cv_scores = []

    for train_idx, val_idx in skf.split(X, y):
        X_train_fold = X.iloc[train_idx]
        X_val_fold = X.iloc[val_idx]
        y_train_fold = y.iloc[train_idx]
        y_val_fold = y.iloc[val_idx]

        model = xgb.XGBClassifier(**params)
        model.fit(
            X_train_fold, y_train_fold,
            eval_set=[(X_val_fold, y_val_fold)],
            verbose=False
        )

        val_preds = model.predict_proba(X_val_fold)[:, 1]
        fold_score = roc_auc_score(y_val_fold, val_preds)
        cv_scores.append(fold_score)

    return np.mean(cv_scores)

# Run optimization
print("Starting hyperparameter optimization with Optuna...")
study = optuna.create_study(direction='maximize',
sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=20, show_progress_bar=True)

print(f"\nBest AUC: {study.best_value:.6f}")
print("Best parameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

# Use best parameters for final model
best_params = study.best_params.copy()
best_params.update({
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'random_state': SEED,
    'early_stopping_rounds': 50
})

[I 2025-08-06 12:13:32,122] A new study created in memory with name: no-name-ddbd2234-b414-4ea6-89b8-359a0c99760f


Starting hyperparameter optimization with Optuna...


Best trial: 0. Best value: 0.964542:   5%|▌         | 1/20 [00:11<03:41, 11.64s/it]

[I 2025-08-06 12:13:43,763] Trial 0 finished with value: 0.9645418020147135 and parameters: {'learning_rate': 0.03574712922600244, 'max_depth': 10, 'subsample': 0.892797576724562, 'colsample_bytree': 0.8394633936788146, 'reg_alpha': 2.5361081166471375e-07, 'reg_lambda': 2.5348407664333426e-07, 'n_estimators': 152}. Best is trial 0 with value: 0.9645418020147135.


Best trial: 1. Best value: 0.967236:  10%|█         | 2/20 [00:22<03:18, 11.04s/it]

...

Best trial: 1. Best value: 0.967236:  70%|███████   | 14/20 [04:07<01:24, 14.11s/it]

[I 2025-08-06 12:17:39,349] Trial 13 finished with value: 0.9662543760006409 and parameters: {'learning_rate': 0.19905093224819492, 'max_depth': 7, 'subsample': 0.9157908265277859, 'colsample_bytree': 0.9237316722462283, 'reg_alpha': 0.007366495045609111, 'reg_lambda': 0.08205000275042881, 'n_estimators': 304}. Best is trial 1 with value: 0.9672359781036884.


Best trial: 1. Best value: 0.967236:  70%|███████   | 14/20 [04:10<01:47, 17.88s/it]

[W 2025-08-06 12:17:42,500] Trial 14 failed with parameters: {'learning_rate': 0.10125985093587661, 'max_depth': 8, 'subsample': 0.94549259525864, 'colsample_bytree': 0.7119963901081219, 'reg_alpha': 0.005373575450092792, 'reg_lambda': 0.04129806952297961, 'n_estimators': 381} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\Users\robkr\anaconda3\envs\ml-env-stable\Lib\site-packages\optuna\study\_optimize.py", line 201, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "C:\Users\robkr\AppData\Local\Temp\ipykernel_400\3027036262.py", line 35, in objective
    model.fit(
  File "c:\Users\robkr\anaconda3\envs\ml-env-stable\Lib\site-packages\xgboost\core.py", line 705, in inner_f
    return func(**kwargs)
           ^^^^^^^^^^^^^^
  File "c:\Users\robkr\anaconda3\envs\ml-env-stable\Lib\site-packages\xgboost\sklearn.py", line 1683, in fit
    self._Booster = train(
                    ^^^^^^
  File "c:\

KeyboardInterrupt: 

In [ ]:
# best params for xgb from optuna

best_params_xgb_smote_before_cv = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'learning_rate': 0.145429672246945018,
    'max_depth': 10,
    'subsample': 0.7895241287104248,
    'colsample_bytree': 0.830923152513491,
    'reg_alpha': 0.9970732440061464,
    'reg_lambda': 8.926472767956915e-07,
    'n_estimators': 827,
    'random_state': SEED
}

In [ ]:
# Stratified KFold Cross-Validation with OOF predictions
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

sampler = SMOTE(random_state=SEED) if IMBLEARN_AVAILABLE else None

X_test = test.drop(columns=['id', 'y'])

n_folds = 5
skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))
cv_scores = []

print(f"Starting {n_folds}-fold cross-validation...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\nFold {fold + 1}/{n_folds}")

    X_train_fold = X.iloc[train_idx]
    X_val_fold = X.iloc[val_idx]
    y_train_fold = y.iloc[train_idx]
    y_val_fold = y.iloc[val_idx]

    # Apply SMOTE only to training fold
    #if IMBLEARN_AVAILABLE:
    #    X_train_fold, y_train_fold = sampler.fit_resample(X_train_fold, y_train_fold)

    model = xgb.XGBClassifier(**best_params)

    model.fit(
        X_train_fold, y_train_fold,
        eval_set=[(X_val_fold, y_val_fold)],
        verbose=False
    )

    oof_preds[val_idx] = model.predict_proba(X_val_fold)[:, 1]

    test_preds += model.predict_proba(X_test)[:, 1] / n_folds

    fold_score = roc_auc_score(y_val_fold, oof_preds[val_idx])
    cv_scores.append(fold_score)
    print(f"Fold {fold + 1} AUC: {fold_score:.6f}")

oof_score = roc_auc_score(y, oof_preds)
print(f"\nOverall OOF AUC: {oof_score:.6f}")
print(f"CV Std: {np.std(cv_scores):.6f}")

predictions = test_preds

Starting 5-fold cross-validation...

Fold 1/5
Fold 1 AUC: 0.964689

Fold 2/5
Fold 2 AUC: 0.962826

Fold 3/5
Fold 3 AUC: 0.962975

Fold 4/5
Fold 4 AUC: 0.964197

Fold 5/5
Fold 5 AUC: 0.963623

Overall OOF AUC: 0.963661
CV Std: 0.000709


In [ ]:
output = pd.DataFrame({
    'id': test.id,
    'y': predictions
})

output.to_csv('attempt4.csv', index=False)
print("Your submission was successfully saved!")

Your submission was successfully saved!


Ensembling

In [ ]:
# ENSEMBLE APPROACH - Multiple Models with Different Strategies
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

sampler = SMOTE(random_state=SEED) if IMBLEARN_AVAILABLE else None

X_test = test.drop(columns=['id', 'y'])

n_folds = 5
skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)

# Initialize prediction arrays for different models
oof_preds_xgb = np.zeros(len(X))
oof_preds_lgb = np.zeros(len(X))
oof_preds_rf = np.zeros(len(X))

test_preds_xgb = np.zeros(len(X_test))
test_preds_lgb = np.zeros(len(X_test))
test_preds_rf = np.zeros(len(X_test))

cv_scores_xgb = []
cv_scores_lgb = []
cv_scores_rf = []

print(f"Starting {n_folds}-fold ensemble cross-validation...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\nFold {fold + 1}/{n_folds}")

    X_train_fold = X.iloc[train_idx]
    X_val_fold = X.iloc[val_idx]
    y_train_fold = y.iloc[train_idx]
    y_val_fold = y.iloc[val_idx]

    # Apply SMOTE only to training fold
    if IMBLEARN_AVAILABLE:
        X_train_fold, y_train_fold = sampler.fit_resample(X_train_fold, y_train_fold)

    # Model 1: XGBoost
    model_xgb = xgb.XGBClassifier(**best_params_xgb)
    model_xgb.fit(
        X_train_fold, y_train_fold,
        eval_set=[(X_val_fold, y_val_fold)],
        verbose=False
    )
    oof_preds_xgb[val_idx] = model_xgb.predict_proba(X_val_fold)[:, 1]
    test_preds_xgb += model_xgb.predict_proba(X_test)[:, 1] / n_folds

    # Model 2: LightGBM
    model_lgb = lgb.LGBMClassifier(
        objective='binary',
        metric='auc',
        learning_rate=0.1,
        max_depth=8,
        num_leaves=64,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=0.1,
        n_estimators=500,
        random_state=SEED,
        verbose=-1
    )
    model_lgb.fit(
        X_train_fold, y_train_fold,
        eval_set=[(X_val_fold, y_val_fold)],
        callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)]
    )
    oof_preds_lgb[val_idx] = model_lgb.predict_proba(X_val_fold)[:, 1]
    test_preds_lgb += model_lgb.predict_proba(X_test)[:, 1] / n_folds

    # Model 3: Random Forest
    model_rf = RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        min_samples_split=10,
        min_samples_leaf=4,
        max_features='sqrt',
        random_state=SEED,
        n_jobs=-1
    )
    model_rf.fit(X_train_fold, y_train_fold)
    oof_preds_rf[val_idx] = model_rf.predict_proba(X_val_fold)[:, 1]
    test_preds_rf += model_rf.predict_proba(X_test)[:, 1] / n_folds

    # Calculate fold scores
    fold_score_xgb = roc_auc_score(y_val_fold, oof_preds_xgb[val_idx])
    fold_score_lgb = roc_auc_score(y_val_fold, oof_preds_lgb[val_idx])
    fold_score_rf = roc_auc_score(y_val_fold, oof_preds_rf[val_idx])

    cv_scores_xgb.append(fold_score_xgb)
    cv_scores_lgb.append(fold_score_lgb)
    cv_scores_rf.append(fold_score_rf)

    print(f"  XGBoost AUC: {fold_score_xgb:.6f}")
    print(f"  LightGBM AUC: {fold_score_lgb:.6f}")
    print(f"  RandomForest AUC: {fold_score_rf:.6f}")

# Individual model OOF scores
oof_score_xgb = roc_auc_score(y, oof_preds_xgb)
oof_score_lgb = roc_auc_score(y, oof_preds_lgb)
oof_score_rf = roc_auc_score(y, oof_preds_rf)

print(f"\nIndividual OOF Scores:")
print(f"XGBoost: {oof_score_xgb:.6f} (±{np.std(cv_scores_xgb):.6f})")
print(f"LightGBM: {oof_score_lgb:.6f} (±{np.std(cv_scores_lgb):.6f})")
print(f"RandomForest: {oof_score_rf:.6f} (±{np.std(cv_scores_rf):.6f})")

# Ensemble strategies
print(f"\nEnsemble Results:")

# 1. Simple Average
ensemble_simple = (oof_preds_xgb + oof_preds_lgb + oof_preds_rf) / 3
score_simple = roc_auc_score(y, ensemble_simple)
print(f"Simple Average: {score_simple:.6f}")

# 2. Weighted Average (weight by performance)
weights = np.array([oof_score_xgb, oof_score_lgb, oof_score_rf])
weights = weights / weights.sum()
ensemble_weighted = (oof_preds_xgb * weights[0] +
                    oof_preds_lgb * weights[1] +
                    oof_preds_rf * weights[2])
score_weighted = roc_auc_score(y, ensemble_weighted)
print(f"Weighted Average: {score_weighted:.6f}")
print(f"Weights: XGB={weights[0]:.3f}, LGB={weights[1]:.3f}, RF={weights[2]:.3f}")

# 3. Best performing combination
best_two_models = np.argsort([oof_score_xgb, oof_score_lgb, oof_score_rf])[-2:]
if 0 in best_two_models and 1 in best_two_models:  # XGB + LGB
    ensemble_best = (oof_preds_xgb + oof_preds_lgb) / 2
    test_ensemble = (test_preds_xgb + test_preds_lgb) / 2
    combo_name = "XGB + LGB"
elif 0 in best_two_models and 2 in best_two_models:  # XGB + RF
    ensemble_best = (oof_preds_xgb + oof_preds_rf) / 2
    test_ensemble = (test_preds_xgb + test_preds_rf) / 2
    combo_name = "XGB + RF"
else:  # LGB + RF
    ensemble_best = (oof_preds_lgb + oof_preds_rf) / 2
    test_ensemble = (test_preds_lgb + test_preds_rf) / 2
    combo_name = "LGB + RF"

score_best = roc_auc_score(y, ensemble_best)
print(f"Best Two Models ({combo_name}): {score_best:.6f}")

# Choose final ensemble
if score_weighted > score_simple and score_weighted > score_best:
    final_predictions = (test_preds_xgb * weights[0] +
                        test_preds_lgb * weights[1] +
                        test_preds_rf * weights[2])
    print(f"\nUsing Weighted Average ensemble for final predictions")
elif score_best > score_simple:
    final_predictions = test_ensemble
    print(f"\nUsing {combo_name} ensemble for final predictions")
else:
    final_predictions = (test_preds_xgb + test_preds_lgb + test_preds_rf) / 3
    print(f"\nUsing Simple Average ensemble for final predictions")

predictions = final_predictions

Starting 5-fold ensemble cross-validation...

Fold 1/5
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[497]	valid_0's auc: 0.962919
  XGBoost AUC: 0.962501
  LightGBM AUC: 0.962919
  RandomForest AUC: 0.951085

Fold 2/5


KeyboardInterrupt: 

In [ ]:
output = pd.DataFrame({
    'id': test.id,
    'y': predictions
})

output.to_csv('ensembling1.csv', index=False)
print("Your submission was successfully saved!")